# 第 6 章实践：线程模型基础


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）Thread Block 内线程总数上限 ______；Warp Size = ______。
2. （选择）Host 启动参数 `block_dim` 与 `pl.simt.block_dim()` 分别表示：
   A. 都是块内线程数<br/>
   B. 逻辑核数 / 块内各维线程数<br/>
   C. 块内线程数 / 逻辑核数
3. （判断）入口函数可以在辅助函数内嵌套调用。（对/错）
4. （填空）块内协作的三段式模式：单线程初始化 → ______ → 全员原子累加 → ______ → 全员读取。

In [ ]:
!cat ./answer/06.06_objective_answer.txt

## 2. 编程实践一（简单）：逐线程向量加法

**任务**：用 SIMT 实现 `out = x + y`（[1, 1024] FP32），THREADS=256，`kernel[None, 4]` 启动。要求：一维全局索引 + 边界判断；与 torch.add 校验（rtol=0, atol=0 可行吗？说明理由）。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：块内计数

**任务**：每个 Block 有 256 线程，每线程向共享 Tile 的 [0,0] 位置 +1（用原子操作），块内屏障后由 0 号线程把结果写入输出 Tensor 的对应块位置。验证：每个输出元素 = 256。

**提示**：套用"初始化 → 屏障 → 原子累加 → 屏障 → 读取"五步模式；`tid == 0` 判断谁写输出。


In [ ]:
# 在此完成实践二
pass

## 4. 编程实践三（困难）：发散代价测量

**任务**：实现两版"把正数翻倍"的 SIMT 函数：A 版用 `if x > 0` 分支；B 版用无显式分支表达（如 `x + pl.simt.max(x, 0.0)`，输入限定为有限值）。输入构造为前半全正、后半全负（Warp 内无分歧）与交替正负（Warp 内充分分歧）两种。在 950 上测量四组耗时（A/B × 无分歧/有分歧），填表并分析。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">版本</th><th style="text-align:left;">无分歧耗时</th><th style="text-align:left;">有分歧耗时</th></tr>
<tr><td>A（分支版）</td><td>【950 待测】</td><td>【950 待测】</td></tr>
<tr><td>B（无分支版）</td><td>【950 待测】</td><td>【950 待测】</td></tr>
</table>


In [ ]:
!cat ./answer/06.06_practice_answer.txt

### 完整参考实现与验收

参考包含三题全部 Kernel 与四组分歧实验。计时明确标为 Host 墙钟口径；每组先校验再预热，并使用 5 次批次均值的中位数。没有预填测量值。

[参考源码](./src/simt_basics.py) 可在 950 上从章节目录执行 `python src/simt_basics.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/simt_basics.py


## 5. 本节小结

三个实践分别覆盖索引映射、块内协作、发散代价——SIMT 编程模型的三大基本功。下一章进入 SIMT 算子开发：用这些基本功实现 Embedding Gather 等真实不规则算子。